In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Historical PM2.5 and wildfire observations
forecast = pd.read_csv(
    project_dir / "data/processed/pm25_forecasting.csv",
    parse_dates=["date", "target_date"]
)

# Historical weather
weather = pd.read_csv(
    project_dir / "data/processed/statewide_weather_daily.csv",
    parse_dates=["date"]
)

# Wind-aligned wildfire evidence
transport = pd.read_csv(
    project_dir / "data/processed/transport_features_daily.csv",
    parse_dates=["date"]
)

print("Forecasting records:", len(forecast))
print("Weather records:", len(weather))
print("Transport records:", len(transport))

print("\nAvailable stations:",
      forecast["station_id"].nunique())


Forecasting records: 8283
Weather records: 8924
Transport records: 8924

Available stations: 97


In [2]:

evidence = forecast.merge(
    weather[
        [
            "station_id", "date",
            "temperature_mean",
            "humidity_mean",
            "wind_speed_mean",
            "precipitation_total",
            "wind_sin_mean",
            "wind_cos_mean"
        ]
    ],
    on=["station_id", "date"],
    how="left",
    validate="one_to_one"
)

evidence = evidence.merge(
    transport[
        ["station_id", "date", "upwind_frp_100km"]
    ],
    on=["station_id", "date"],
    how="left",
    validate="one_to_one"
)

assert not evidence.duplicated(
    ["station_id", "date"]
).any()

print("Evidence store ready:", len(evidence))


Evidence store ready: 8283


In [3]:

tool_log = []

def get_record(station_id, date):
    date = pd.Timestamp(date).normalize()

    record = evidence[
        (evidence["station_id"] == station_id)
        & (evidence["date"] == date)
    ]

    if record.empty:
        raise ValueError(
            "No observation for that station and date."
        )

    return record.iloc[0]


def fire_tool(station_id, date):
    row = get_record(station_id, date)

    tool_log.append({
        "tool": "fire",
        "station_id": station_id,
        "date": str(date)
    })

    return {
        "fire_count_50km": int(row["fire_count_50km"]),
        "fire_count_100km": int(row["fire_count_100km"]),
        "frp_sum_100km": float(row["frp_sum_100km"]),
        "source": "NASA FIRMS VIIRS"
    }


def weather_tool(station_id, date):
    row = get_record(station_id, date)

    tool_log.append({
        "tool": "weather",
        "station_id": station_id,
        "date": str(date)
    })

    return {
        "temperature": float(row["temperature_mean"]),
        "humidity": float(row["humidity_mean"]),
        "wind_speed": float(row["wind_speed_mean"]),
        "precipitation": float(row["precipitation_total"]),
        "wind_sin": float(row["wind_sin_mean"]),
        "wind_cos": float(row["wind_cos_mean"]),
        "source": "Open-Meteo historical reanalysis"
    }


def transport_tool(station_id, date):
    row = get_record(station_id, date)

    tool_log.append({
        "tool": "transport",
        "station_id": station_id,
        "date": str(date)
    })

    return {
        "upwind_frp_100km": float(
            row["upwind_frp_100km"]
        ),
        "source": "EcoSentinel derived transport feature"
    }


In [4]:

station_id = "06-051-0005"
date = "2025-09-07"

tool_log.clear()

print("FIRE EVIDENCE")
print(fire_tool(station_id, date))

print("\nWEATHER EVIDENCE")
print(weather_tool(station_id, date))

print("\nTRANSPORT EVIDENCE")
print(transport_tool(station_id, date))

print("\nTOOL EXECUTION LOG")
display(pd.DataFrame(tool_log))


FIRE EVIDENCE
{'fire_count_50km': 0, 'fire_count_100km': 1, 'frp_sum_100km': 1.39, 'source': 'NASA FIRMS VIIRS'}

WEATHER EVIDENCE
{'temperature': 16.45, 'humidity': 38.958333333333336, 'wind_speed': 6.0874999999999995, 'precipitation': 0.0, 'wind_sin': -0.2271761876457934, 'wind_cos': -0.2918843733629191, 'source': 'Open-Meteo historical reanalysis'}

TRANSPORT EVIDENCE
{'upwind_frp_100km': 0.3640044791501624, 'source': 'EcoSentinel derived transport feature'}

TOOL EXECUTION LOG


,tool,station_id,date
0,fire,06-051-0005,2025-09-07
1,weather,06-051-0005,2025-09-07
2,transport,06-051-0005,2025-09-07
